# Referrals Exploratory Data Analysis

## Purpose

This notebook checks referral data. It checks missing values, IDs, dates, status values, and links to follow-ups.

## Files used

- `Referrals.csv` — referral records
- `Follow_Ups.csv` — follow-up records for referrals
- `Risk_Assessments.csv` — risk records linked to referrals

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

def find_raw_data_dir(start: Path = Path.cwd()) -> Path:
    for directory in (start, *start.parents):
        for candidate in (directory / 'data' / 'raw', directory / 'data-analytics' / 'data' / 'raw'):
            if candidate.is_dir():
                return candidate
    raise FileNotFoundError('Could not locate data-analytics/data/raw')

RAW_DATA_DIR = find_raw_data_dir()

## 1. Load the data

In [ ]:
referrals = pd.read_csv(RAW_DATA_DIR / 'Referrals.csv')
follow_ups = pd.read_csv(RAW_DATA_DIR / 'Follow_Ups.csv')
risk_assessments = pd.read_csv(RAW_DATA_DIR / 'Risk_Assessments.csv')
pd.DataFrame({'dataset': ['Referrals', 'Follow Ups', 'Risk Assessments'], 'rows': [len(referrals), len(follow_ups), len(risk_assessments)], 'columns': [len(referrals.columns), len(follow_ups.columns), len(risk_assessments.columns)]})

## 2. Check the file

In [ ]:
referrals

In [ ]:
referral_profile = pd.DataFrame({
    'data_type': referrals.dtypes.astype(str),
    'missing_count': referrals.isna().sum(),
    'unique_values': referrals.nunique(dropna=False),
})
referral_profile

In [ ]:
required_columns = ['referral_id', 'risk_assessment_id', 'referral_type', 'urgency', 'status', 'referred_at', 'due_date']
assert set(required_columns).issubset(referrals.columns)
assert referrals[required_columns].notna().all().all()
print('All required referral fields have values.')

## 3. Check IDs and duplicates

In [ ]:
id_checks = pd.Series({
    'duplicate_referral_ids': referrals['referral_id'].duplicated().sum(),
    'wrong_referral_id_format': (~referrals['referral_id'].str.match(r'^REF-[0-9]{3}$', na=False)).sum(),
    'wrong_risk_assessment_id_format': (~referrals['risk_assessment_id'].str.match(r'^RSK-[0-9]{4}$', na=False)).sum(),
    'duplicate_risk_assessment_ids': referrals['risk_assessment_id'].duplicated().sum(),
    'exact_duplicate_rows': referrals.duplicated().sum(),
})
id_checks.to_frame('count')

In [ ]:
assert id_checks.eq(0).all()
print('Referral IDs and risk assessment IDs are unique and correctly formatted.')

## 4. Check dates

In [ ]:
referral_dates = referrals.copy()
for column in ['referred_at', 'due_date']:
    referral_dates[column] = pd.to_datetime(referral_dates[column], utc=True, errors='coerce')
date_checks = pd.Series({
    'bad_referred_dates': referral_dates['referred_at'].isna().sum(),
    'bad_due_dates': referral_dates['due_date'].isna().sum(),
    'due_before_referral': (referral_dates['due_date'] < referral_dates['referred_at']).sum(),
})
date_checks.to_frame('count')

In [ ]:
assert date_checks.eq(0).all()
print('All referral dates are valid. Every due date is after its referral date.')

## 5. Review types, urgency, and status

In [ ]:
for column in ['referral_type', 'urgency', 'status']:
    display(referrals[column].value_counts(dropna=False).to_frame('referral_count'))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, column, color in zip(axes, ['referral_type', 'urgency', 'status'], ['#176B87', '#64CCC5', '#DAA520']):
    referrals[column].value_counts().plot(kind='bar', ax=ax, color=color, title=column.replace('_', ' ').title())
    ax.set_xlabel('')
    ax.set_ylabel('Referral count')
    ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

## 6. Check risk assessment links

In [ ]:
unknown_risk_assessment_ids = sorted(set(referrals['risk_assessment_id']) - set(risk_assessments['risk_assessment_id']))
risk_coverage = risk_assessments[['risk_assessment_id', 'risk_level', 'requires_referral']].assign(
    has_referral=lambda frame: frame['risk_assessment_id'].isin(referrals['risk_assessment_id'])
)
required_risks = risk_coverage[risk_coverage['requires_referral']]
invalid_referrals = risk_coverage[risk_coverage['has_referral'] & ~risk_coverage['requires_referral']]
display(required_risks.groupby(['risk_level', 'has_referral']).size().rename('risk_count').reset_index())
print('Unknown risk assessment IDs:', len(unknown_risk_assessment_ids))
print('Referrals linked to risks that do not require referral:', len(invalid_referrals))
print('Required referrals with no referral row:', (~required_risks['has_referral']).sum())
print('Referral coverage:', f"{required_risks['has_referral'].mean():.1%}")

In [ ]:
assert not unknown_risk_assessment_ids
assert invalid_referrals.empty
print('All referrals link to known risks that require referral.')

## 7. Check follow-up coverage

In [ ]:
unknown_follow_up_referrals = sorted(set(follow_ups['referral_id']) - set(referrals['referral_id']))
referral_coverage = referrals[['referral_id', 'status']].assign(
    has_follow_up=lambda frame: frame['referral_id'].isin(follow_ups['referral_id'])
)
display(referral_coverage.groupby(['status', 'has_follow_up']).size().rename('referral_count').reset_index())
print('Unknown referral IDs in follow-ups:', len(unknown_follow_up_referrals))
print('Referrals without a follow-up:', (~referral_coverage['has_follow_up']).sum())

In [ ]:
assert not unknown_follow_up_referrals
print('All follow-ups link to a known referral.')

## 8. Findings

There are 12 referrals. All required fields have values. Referral IDs and risk assessment IDs are unique and correctly formatted. All dates are valid.

There are 7 general practitioner referrals, 2 dietitian referrals, 2 counsellor referrals, and 1 fitness coach referral. There are 11 priority referrals and 1 urgent referral.

Eight referrals are complete, 2 are scheduled, and 2 are issued. Eight referrals have follow-up records. Four referrals do not have follow-up records yet.

All 12 referrals link to known risk assessments that require referral. There are 37 risk rows that require referral. Only 12 have referral rows. This leaves 25 required referrals without referral records. Referral coverage is 32.4%.

## Next steps

- Review the 25 risk rows that require referral but have no referral record.
- Keep the ID, date, duplicate, and follow-up checks.
- Check referrals that are issued or scheduled and do not yet have follow-ups.
- Add more data before looking for trends.